"""
Baseline clínico vs TFs vs clínico + TFs (TCGA-KIRC, casos completos)
Responde a: R2.M3 (valor independente dos fatores clinicopatológicos),
            R3.8 (comparar com fatores de risco estabelecidos), R3.12 (baselines)

Requer ml_pipeline_revision.py na mesma pasta (reaproveita load_data,
make_strata, boruta_select, fit_predict, all_metrics, paired_auc_diff e a
configuração: SEED, BALANCING, STRAT_COL, TEST_SIZE, N_OUTER, OUT_DIR).

Modelos, sempre treinados e avaliados nos MESMOS pacientes:
  clinico      regressão logística com grau + pT + idade
  TFs          Random Forest com os TFs selecionados pelo Boruta (como no pipeline)
  clinico_TFs  regressão logística com grau + pT + idade + escore de TFs
               (escore = logito da probabilidade do RF; no treino ele é
               out-of-fold, para o modelo combinado não ver predições superotimistas)

Saídas em <OUT_DIR>/clinico/:
  resultados_clinico_<BALANCING>.json, regressao_multivariavel.csv, predicoes_*.csv
"""

In [1]:
import json
import os
 
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
 
import ml_pipeline_revision as P

In [2]:
AGE_COL = "age.at.index"
PT_COL = "ajcc_pathologic_T"
GRADE_COL = P.STRAT_COL
OUT = os.path.join(P.OUT_DIR, "clinico")
os.makedirs(OUT, exist_ok=True)

In [3]:
def find_col(meta, name):
    """Acha a coluna tolerando diferença de maiúsculas e de '_' vs '.'."""
    if name in meta.columns:
        return name
    norm = lambda s: s.lower().replace("_", ".")
    matches = [c for c in meta.columns if norm(c) == norm(name)]
    if not matches:
        raise KeyError(f"coluna '{name}' não encontrada. Disponíveis: {list(meta.columns)}")
    return matches[0]
 
 
def clinical_matrix(meta):
    gc, tc, ac = find_col(meta, GRADE_COL), find_col(meta, PT_COL), find_col(meta, AGE_COL)
    grade = meta[gc].astype(str).str.extract(r"G([1-4])", expand=False).astype(float)
    pt = meta[tc].astype(str).str.extract(r"T([1-4])", expand=False).astype(float)  # T1a, T3b... -> 1, 3
    age = pd.to_numeric(meta[ac], errors="coerce")
    if age.median() < 0:  # idade em dias negativos (days_to_birth)
        age = age.abs() / 365.25
    return pd.DataFrame({"grade": grade, "pT": pt, "age": age}, index=meta.index)
 
 
def load_complete_cases():
    X, y, meta, flow = P.load_data()
    C = clinical_matrix(meta)
    ok = C.notna().all(axis=1)
    excl = meta.loc[~ok]
    flow = dict(flow)
    flow["excluidos_sem_grau_pT_ou_idade"] = int((~ok).sum())
    flow["excluidos_M1"] = int(y.loc[~ok].sum())
    X, y, meta, C = X.loc[ok], y.loc[ok], meta.loc[ok], C.loc[ok]
    flow["casos_completos"] = int(len(y))
    flow["casos_completos_M0"], flow["casos_completos_M1"] = int((y == 0).sum()), int(y.sum())
    print("Casos completos:", flow)
    return X, y, meta, C, flow